### 1. Initialization

Imports and necessary parameters.

In [69]:
## ----- IPYTHON COMMANDS ----- ##
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [70]:
## ----- IMPORTS ----- ##
import sys
sys.path.insert(0, './param_import/')
from imports import *
import parameters as pm
import time


### 2. Creating new catalog

We are creating a catalog using pandas DataFrames and based on the previous *satellite_constellation_catalog.csv* file; essentially we replicate everything there taking into account the different satellites that appear on the specific data that we are using. 

This should be repeated for every choice of data, beam model, and frequency range (but irrelevant for masking).

In [3]:
# getting individual satellites that appear
start = time.perf_counter()
f2 = pickle.load(open("{}{}_individual_satellite_angular_position_{}_beam_{}_{}MHz.p".format(
    pm.path_data,str(pm.block),pm.beam_model,pm.fs,pm.fe),"rb",), encoding="latin1")
elapsed = time.perf_counter() - start
print(f"This took {elapsed:.2f} seconds, or {elapsed/60:.2f} minutes".format())


This took 82.34 seconds, or 1.37 minutes


In [83]:
# creating these arrays
satellite = []
constellation = []

# looping through the dictionary keys
for cons in f2.keys():
    sats = list(f2[cons].keys())
    satellite.extend(sats)
    constellation.extend(np.full(len(sats),cons))

# creating temporary dataframe
cat_sat = pd.DataFrame({"Sat":satellite, "Sys":constellation})


In [87]:
# getting old catalog
catalog = pd.read_csv(pm.path_catalog)

# cleaning up the data
catalog.loc[catalog["Sys"]=="beidou-2", "Sys"] = "beidou"
catalog.loc[catalog["Sys"]=="beidou-3", "Sys"] = "beidou"


In [86]:
# creating new dataframe
new_catalog = cat.merge(cat_sat, on="Sys")
col = new_catalog.pop("Sat")  # <-- putting Sat column first
new_catalog.insert(0, "Sat", col)
new_catalog = new_catalog.sort_values("Sat", key = lambda x: x.map({v:i for i, v in enumerate(col)}))
new_catalog = new_catalog.reset_index(drop=True)

# saving data
new_catalog.to_csv(pm.path_new_catalog, index=False)


### 3. Saving matrix information  in a different format

Currently the file takes 2 minutes to load; saving only as a dictionary (we know the constellation of each satellite from the csv file already) and saving as arrays instead of masked arrays saves a lot of time.

In [88]:
# creating dictionary (information on constellations is not needed)
satellite_data = {sat: f2[cons][sat].filled(0) for cons in f2 for sat in f2[cons]}

# saving data
fname = "{}_satellite_angular_positions_{}_{}-{}.npz".format(str(pm.block),pm.beam_model,pm.fs,pm.fe)
with open(fname, "wb") as f:
    pickle.dump(satellite_data, f)


In [82]:
# trying to retrieve the information, much faster!
start = time.perf_counter()
f3 = pickle.load(open(fname,"rb",), encoding="latin1")
elapsed = time.perf_counter() - start
print(f"This took {elapsed:.2f} seconds, or {elapsed/60:.2f} minutes".format())


This took 2.34 seconds, or 0.04 minutes
